## Supplementary Figure 5 — In-field vs. out-of-field firing rate

Tests whether the superficial spatial-coding phenotype reflects a genuine loss of
spatial selectivity or simply elevated, non-specific excitability
(Reviewer 3, minor point 2).

For every pyramidal cell with at least one detected place field we compute, from
the speed-filtered rate map:

* **in-field rate**  = spikes inside the field(s)  / dwell-time inside  (Hz)
* **out-field rate** = spikes outside the field(s) / dwell-time outside (Hz)
* **in/out ratio**   = in-field / out-field  (spatial signal-to-noise)

Computed on **all pyramidal cells** (not only place cells) to avoid conditioning
on the outcome. The in/out-field columns were pre-computed from the NWB position
data (`XY_mid_brain`) by `compute_infield_outfield.py` and stored in the table
loaded below; the `*_units_table_withDLC.pkl` files contain spikes but **not** the
animal trajectory, so position is taken from the NWB.

Statistics: linear mixed model `value ~ genotype + (1 | animal)` (animal as the
unit). Plots are SuperPlots: faint violin = cell distribution, small dots = cells
(blue = CR;DTA-, red = CR;DTA+), large dots = per-animal means, black bar =
mean +/- SEM across mice.

In [ ]:
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore")

# ---- config ----
FONT_SIZE = 7
TEXT_KWARGS = {"fontsize": FONT_SIZE, "color": "black"}
plt.rcParams.update({
    "font.size": FONT_SIZE, "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "sans-serif"],
    "axes.linewidth": 0.8, "xtick.major.width": 0.8, "ytick.major.width": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
BASE = {"control": "#0000FF", "exp": "#FF0000"}
CONTROL_IDS = ["65165", "65091", "63383", "66539", "65622"]

TABLE = "/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/functional_properties_with_python_measurements_pycirc_infield.pkl"
SAVE = "/Users/sachuriga/Desktop/Projects/CR_CA1_paper/Figures_neuron_report_raw/suppfig5_in_out_field.pdf"


def mixedlm_p(d, var, log=True):
    """LMM p for genotype with animal as random intercept."""
    d = d[["animal_id", "group_ani", var]].copy()
    d["y"] = pd.to_numeric(d[var], errors="coerce")
    d = d.dropna(subset=["y"])
    if log:
        d = d[d["y"] > 0]; d["y"] = np.log(d["y"])
    d["g"] = pd.Categorical(d["group_ani"], categories=["control", "exp"])
    try:
        return smf.mixedlm("y ~ g", d, groups=d["animal_id"]).fit(reml=True).pvalues.get("g[T.exp]", np.nan)
    except Exception:
        return np.nan


def superplot(ax, d, var, title, log=True, ylim=None):
    order = ["control", "exp"]; xmap = {"control": 0, "exp": 1}
    d = d.dropna(subset=[var]).copy()
    sns.violinplot(data=d, x="group_ani", y=var, order=order, ax=ax, hue="group_ani",
                   palette=BASE, cut=0, inner=None, linewidth=0, legend=False)
    for c in ax.collections:
        c.set_alpha(0.12)
    for g in order:
        sub = d[d.group_ani == g]
        for a in sub["animal_id"].unique():
            yy = pd.to_numeric(sub[sub.animal_id == a][var], errors="coerce").dropna().values
            if len(yy) == 0:
                continue
            ax.scatter(np.random.normal(xmap[g], 0.06, len(yy)), yy, s=2.5,
                       color=BASE[g], alpha=0.30, linewidth=0, zorder=2)
            ax.scatter(xmap[g], np.mean(yy), s=24, color=BASE[g], edgecolor="k", linewidth=0.6, zorder=4)
    for g in order:
        am = d[d.group_ani == g].groupby("animal_id")[var].mean().values
        if len(am):
            ax.errorbar(xmap[g], np.mean(am), yerr=np.std(am) / np.sqrt(len(am)),
                        fmt="_", color="k", capsize=3, markersize=10, zorder=5, elinewidth=1.0)
    p = mixedlm_p(d, var, log)
    ax.set_ylabel(title, **TEXT_KWARGS); ax.set_xlabel("")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["CR;DTA-", "CR;DTA+"], rotation=-45, ha="left", rotation_mode="anchor", **TEXT_KWARGS)
    if ylim is not None:
        ax.set_ylim(*ylim)                      # fixed visible range (points outside are clipped, not removed from stats)
    ylo, yhi = ax.get_ylim()
    if pd.notna(p) and p < 0.05:
        bh = (yhi - ylo) * 0.03
        y0 = ylo + (yhi - ylo) * 0.88           # bracket near top of the visible axis
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*"
        ax.plot([0, 0, 1, 1], [y0, y0 + bh, y0 + bh, y0], color="k", lw=0.8)
        ax.text(0.5, y0 + bh * 2.0, f"{sig} p={p:.3f}", ha="center", va="center", **TEXT_KWARGS)
    else:
        ax.set_title(f"p = {p:.3f}", fontsize=6)
    sns.despine(ax=ax)
    return p


# ---- load pre-computed table ----
df = pd.read_pickle(TABLE)
df = df[(df["buzaki_py_cell_type"] == "pyramidal") & (df["session"] == "A")].copy()
df["animal_id"] = df["animal_id"].astype(str)
df["group_ani"] = np.where(df["animal_id"].isin(CONTROL_IDS), "control", "exp")
fd = df[df["in_out_ratio"].notna()]   # pyramidal cells with >=1 detected field

# ---- figure: rows = deep / superficial, cols = in / out / ratio ----
metrics = [("in_field_rate", "In-field rate (Hz)"),
           ("out_field_rate", "Out-field rate (Hz)"),
           ("in_out_ratio", "In/out ratio (S/N)")]
fig, axes = plt.subplots(2, 3, figsize=(6.0, 5.2))
for r, sp in enumerate(["deep", "superficial"]):
    sub = fd[fd.sub_population == sp]
    for c, (v, t) in enumerate(metrics):
        superplot(axes[r, c], sub, v, t, ylim=((0, 150) if v == "in_out_ratio" else None))
    n_cell = len(sub); n_mice = sub["animal_id"].nunique()
    axes[r, 0].annotate(f"{sp.capitalize()}\n({n_cell} cells, {n_mice} mice)",
                        xy=(-0.55, 0.5), xycoords="axes fraction", rotation=90,
                        va="center", ha="center", fontsize=8, fontweight="bold")

fig.suptitle("Supplementary Figure 5 — In- vs out-of-field firing (all pyramidal cells)",
             fontsize=9)
fig.tight_layout()
import os
os.makedirs(os.path.dirname(SAVE), exist_ok=True)
fig.savefig(SAVE, transparent=True, bbox_inches="tight")
plt.show()
print("saved ->", SAVE)
